# DS102: Thực nghiệm thu thập dữ liệu Books to Scrape

Mục đích:
1. Khảo sát cấu trúc HTML và kiểm tra parser bằng BeautifulSoup.
2. Kiểm tra tính toàn vẹn của tập dữ liệu 1.000 cuốn sách thu thập được.
3. Phân tích thống kê mô tả (giá, phân phối sao) chuẩn bị cho các mô hình Machine Learning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3

# Đường dẫn tới thư mục data của Bài 2
root = Path.cwd()
if (root / 'Bài 2').exists():
    data_dir = root / 'Bài 2' / 'data'
elif (root.parent / 'Bài 2').exists():
    data_dir = root.parent / 'Bài 2' / 'data'
else:
    data_dir = Path('../Bài 2/data')

print("Thư mục dữ liệu:", data_dir.resolve())
csv_file = data_dir / 'books.csv'
db_file = data_dir / 'books.db'

df = pd.read_csv(csv_file)
print(f"Tổng số cuốn sách: {len(df)}")
df.head()

## 2. Kiểm tra chất lượng dữ liệu (Data Quality Check)
- Kiểm tra giá trị khuyết thiếu (missing values).
- Kiểm tra trùng lặp đường dẫn (URL duplicated).
- Kiểm tra kiểu dữ liệu các cột.

In [ ]:
print("--- KIỂM TRA MISSING VALUES ---")
print(df.isna().sum())
print("\n--- KIỂM TRA TRÙNG LẶP URL ---")
print("Số URL trùng:", df['product_url'].duplicated().sum())
print("\n--- THÔNG TIN KIỂU DỮ LIỆU ---")
df.info()

## 3. Thống kê mô tả (Descriptive Statistics)
- Phân phối biến liên tục: Giá sách (`price_gbp`).
- Phân phối biến thứ bậc: Số sao đánh giá (`rating_star`).
- Tỷ lệ còn hàng (`is_in_stock`).

In [ ]:
print("--- THỐNG KÊ GIÁ SÁCH (£) ---")
display(df[['price_gbp', 'rating_star']].describe())

print("\n--- PHÂN PHỐI SỐ SAO ---")
print(df['rating_star'].value_counts().sort_index())

## 4. Truy vấn từ Cơ sở dữ liệu SQLite
Kiểm tra việc lưu trữ vào SQLite (`books.db`).

In [ ]:
conn = sqlite3.connect(str(db_file))
query = """
SELECT rating_star, COUNT(*) as count, ROUND(AVG(price_gbp), 2) as avg_price
FROM books
GROUP BY rating_star
ORDER BY rating_star DESC;
"""
sql_df = pd.read_sql_query(query, conn)
conn.close()
display(sql_df)